# Chapter 3 — the MLP: embeddings, context, and how to actually train

From *Neural Networks: Zero to Hero — The Textbook*.

Run each cell with **Shift+Enter**. Before you run one, say out loud what you expect it to print; being wrong is the useful part.


## Chapter 3 — the MLP: embeddings, context, and how to actually train

**Video:** 1h15m · [youtu.be/TCH_1BHY58I](https://youtu.be/TCH_1BHY58I) · **Based on:** Bengio et al. 2003, "A Neural Probabilistic Language Model," the ancestor of everything that followed. [transcript]

### The problem

One letter of context is not enough, and a lookup table cannot hold more (205 trillion cells, Chapter 2). You need a model that generalizes to contexts it has never seen.

### The central idea: embeddings

Give every character a short list of learned numbers, say 2 or 10 of them, called an **embedding**. These numbers are parameters, so training decides what they should be. Characters used in similar ways drift toward similar numbers.

**Why this defeats the table explosion.** A table treats `aeb` and `aob` as unrelated cells with unrelated counts. If `e` and `o` end up with similar embeddings, a context the model has never seen gets handled sensibly by analogy with one it has. Information is *shared* rather than memorized.

**Analogy.** A table is a phone book: to know anything about a name, that exact name must be listed. Embeddings give every person coordinates instead (age, city, profession), so you can make reasonable guesses about someone you have never met, because they sit near people you have.

> **Say it to a six-year-old.** Instead of remembering every single word you ever heard, you notice that "cat" and "dog" are both furry animals, so they go in the same part of your head. Then when you hear about a new animal you have never met, you already have a good guess about it, because you put it near the other furry ones.

> **For the PhD in the room.** This is distributed representation in Hinton's sense, and Bengio's 2003 paper is the one that made it work for language: the count table's parameters grow as V^n while the MLP's grow as V·d + n·d·h, trading exponential blowup for a fixed-capacity bottleneck. The embedding matrix is exactly a learned linear map from the one-hot simplex, so "look up row i" and "multiply by a one-hot vector" are the same operation, which is why nobody implements it as a matrix multiply. Note what is *not* happening here: no factorization objective, no explicit similarity loss. Similarity structure appears only because it lowers next-character cross-entropy, which is the same argument later made for word2vec and, ultimately, for why language model representations transfer at all.

### Step 1 — build the dataset with a sliding window

**Run it.**

In [ ]:
import torch
words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}; stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

block_size = 3   # how many characters we use to predict the next one

def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size          # start padded with '...'
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)               # the 3 characters we see
            Y.append(ix)                    # the character that follows
            context = context[1:] + [ix]    # slide the window forward
    return torch.tensor(X), torch.tensor(Y)

# show the mechanism on one word
X, Y = build_dataset(['emma'])
for x, y in zip(X, Y):
    print(''.join(itos[i.item()] for i in x), '-->', itos[y.item()])

**What you should see:**

**Expected output:**

```
... --> e
..e --> m
.em --> m
emm --> a
mma --> .
```

[verified]

The window slides one character at a time, and the `.` padding at the start lets the model handle the first characters of a name with the same machinery as the middle.

### Step 2 — split into train, dev, test

**Run it.**

In [ ]:
import random
random.seed(42)
random.shuffle(words)
n1, n2 = int(0.8*len(words)), int(0.9*len(words))
Xtr,  Ytr  = build_dataset(words[:n1])      # 80%
Xdev, Ydev = build_dataset(words[n1:n2])    # 10%
Xte,  Yte  = build_dataset(words[n2:])      # 10%
print("train:", tuple(Xtr.shape), "dev:", tuple(Xdev.shape), "test:", tuple(Xte.shape))

**What you should see:**

**Expected output:**

```
train: (182625, 3) dev: (22655, 3) test: (22866, 3)
```

[verified]

182,625 training examples. **Note the shuffle happens on words, not on examples.** Splitting by example would leak: two examples from the same name share context, so the same name could appear in both training and validation, and your validation number would be a lie.

### Step 3 — the network

**Run it.**

In [ ]:
import torch.nn.functional as F
g = torch.Generator().manual_seed(2147483647)
n_embd, n_hidden = 10, 200

C  = torch.randn((27, n_embd),            generator=g)   # embedding table
W1 = torch.randn((n_embd*block_size, n_hidden), generator=g)
b1 = torch.randn(n_hidden,                generator=g)
W2 = torch.randn((n_hidden, 27),          generator=g)
b2 = torch.randn(27,                      generator=g)
parameters = [C, W1, b1, W2, b2]
print("total parameters:", sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

**What you should see:**

**Expected output:**

```
total parameters: 11897
```

[verified]

Where 11,897 comes from: `C` is 27×10 = 270. `W1` is 30×200 = 6,000, plus 200 biases. `W2` is 200×27 = 5,400, plus 27 biases. Total 270 + 6,000 + 200 + 5,400 + 27 = **11,897**.

**Run it.** The forward pass, one piece at a time, so you can see every shape:

In [ ]:
ix = torch.randint(0, Xtr.shape[0], (32,), generator=g)   # a mini-batch of 32
emb = C[Xtr[ix]]
print("1. embeddings:      ", tuple(emb.shape), "  <- 32 examples, 3 chars, 10 numbers each")
flat = emb.view(-1, n_embd*block_size)
print("2. flattened:       ", tuple(flat.shape), "     <- the 3 chars glued into one 30-vector")
hpreact = flat @ W1 + b1
print("3. pre-activation:  ", tuple(hpreact.shape), "    <- 200 hidden neurons")
h = torch.tanh(hpreact)
print("4. after tanh:      ", tuple(h.shape))
logits = h @ W2 + b2
print("5. logits:          ", tuple(logits.shape), "     <- one score per possible next char")
loss = F.cross_entropy(logits, Ytr[ix])
print("6. loss:            ", round(loss.item(), 4))

**What you should see:**

**Expected output:**

```
1. embeddings:       (32, 3, 10)   <- 32 examples, 3 chars, 10 numbers each
2. flattened:        (32, 30)      <- the 3 chars glued into one 30-vector
3. pre-activation:   (32, 200)     <- 200 hidden neurons
4. after tanh:       (32, 200)
5. logits:           (32, 27)      <- one score per possible next char
6. loss:             27.8817
```

[verified]

**`C[Xtr[ix]]` deserves a pause.** `Xtr[ix]` is a 32×3 tensor of integers. Indexing the 27×10 table `C` with it produces a 32×3×10 tensor: PyTorch looked up all 96 characters at once and stacked the results. This is **fancy indexing**, and it is doing the job that a one-hot matrix multiply did in Chapter 2, without the wasted arithmetic.

**`.view(-1, 30)` deserves another.** It reinterprets the same block of memory with a different shape. The `-1` means "work this dimension out for me," here 32. It is nearly free because nothing is copied. Getting this wrong, for example using `.view(30, -1)`, produces a valid tensor of garbage and no error message.

**And that loss of 27.88 is a red flag.** Section 1.12 says a fresh 27-way model should score 3.2958. It scores 27.88, meaning it starts out confidently wrong. That is Chapter 4's subject; note it and move on.

### Step 4 — mini-batches

Computing gradients on all 182,625 examples per step is accurate and slow. Instead, sample 32 random examples per step. The gradient direction is noisy, "not the actual gradient direction, but the gradient direction is good enough" [transcript], and you get to take a hundred times more steps in the same wall-clock time.

**Analogy.** An approximate step taken now beats a perfect step taken next week.

> **For the PhD in the room.** This is stochastic gradient descent, and the noise is not purely a cost: minibatch gradient noise acts as an implicit regularizer, with a well-documented relationship between the noise scale and the learning rate to batch size ratio (Smith and Le, 2018). The practical consequence is that increasing the batch size without rescaling the learning rate changes the effective regularization, which is why "just use a bigger batch" often loses accuracy.

### Step 5 — find the learning rate by experiment

Rather than guessing, sweep it across orders of magnitude and look at the result.

**Run it.**

In [ ]:
# a trimmed version: 2000 steps at each learning rate, then measure the full training loss
for lr in [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]:
    ...   # rebuild the network, train 2000 steps at this lr, then evaluate

**What you should see** (the full script is in the exercises; these are the measured results):

**Expected output:**

```
lr=0.0001   training loss after 2000 steps: 3.1873
lr=0.001    training loss after 2000 steps: 2.7939
lr=0.01     training loss after 2000 steps: 2.5179
lr=0.1      training loss after 2000 steps: 2.3709
lr=1.0      training loss after 2000 steps: 2.4446
lr=10.0     training loss after 2000 steps: 87.3967
```

[verified]

Read that table and you have the whole learning-rate story:

- **0.0001** barely moves off the starting point. The lecture's phrase for this is "the loss is barely decreasing" [transcript].
- **0.1** is the sweet spot.
- **1.0** is past the sweet spot; it still trains, but worse.
- **10.0** diverges spectacularly to 87, far worse than random guessing at 3.2958. The steps overshoot the valley so badly that the parameters explode.

**Learning rate decay.** Once the loss stops improving, cut the rate by 10× and continue. Large steps get you near the valley quickly; small steps settle into it. This is worth roughly 0.05 to 0.1 of loss in this model. [verified]

### Step 6 — train properly and evaluate

**Run it.**

In [ ]:
for i in range(30000):
    ix = torch.randint(0, Xtr.shape[0], (32,), generator=g)
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1, n_embd*block_size) @ W1 + b1)
    loss = F.cross_entropy(h @ W2 + b2, Ytr[ix])
    for p in parameters:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < 18000 else 0.01        # decay late in training
    for p in parameters:
        p.data += -lr * p.grad

@torch.no_grad()                            # no gradients needed when only measuring
def split_loss(X, Y):
    emb = C[X]
    h = torch.tanh(emb.view(-1, n_embd*block_size) @ W1 + b1)
    return F.cross_entropy(h @ W2 + b2, Y).item()

print(f"train {split_loss(Xtr, Ytr):.4f} | dev {split_loss(Xdev, Ydev):.4f}")

**What you should see:**

**Expected output:**

```
train 2.2618 | dev 2.2778
```

[verified]

Better than the bigram's 2.4544. And read the two numbers together: train 2.2618, dev 2.2778, nearly equal. **That means the model is not memorizing; it is too small.** Overfitting would show as train far below dev. This diagnostic, comparing the two rather than staring at one, is the single most useful habit in the chapter.

**The `@torch.no_grad()` decorator** tells PyTorch not to build a computation graph, since we are only measuring. It roughly halves memory and speeds evaluation up. Forgetting it is harmless but wasteful.

### Step 7 — sanity check by deliberately overfitting

Before a long run, prove the model *can* learn by making it memorize a tiny dataset.

**Run it.**

In [ ]:
# train on only the first 32 examples, for 1000 steps
Xsmall, Ysmall = Xtr[:32], Ytr[:32]
for i in range(1000):
    emb = C[Xsmall]
    h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
    loss = F.cross_entropy(h @ W2 + b2, Ysmall)
    for p in parameters: p.grad = None
    loss.backward()
    for p in parameters: p.data += -0.1 * p.grad
print("loss on 32 memorized examples:", round(loss.item(), 6))

**What you should see:**

**Expected output:**

```
loss on 32 memorized examples: 0.252175
```

[verified]

0.25 against a know-nothing baseline of 3.2958, and still falling. Run 5,000 steps instead of 1,000 and it approaches zero, because 11,897 parameters can trivially memorize 32 examples. (It does not get there in 1,000 steps here because this network still has the broken initialization from step 3, which is the next section's subject.)

If the loss *cannot* be driven down on a tiny batch, you have a bug, and you found it in 20 seconds instead of after an hour of real training. This is a unit test, not a result. [transcript]

### Step 8 — sample from it

**Run it.**

In [ ]:
g = torch.Generator().manual_seed(2147483647 + 10)
for _ in range(8):
    out, context = [], [0] * block_size
    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        probs = F.softmax(h @ W2 + b2, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        if ix == 0: break
        out.append(itos[ix])
    print(''.join(out))

**What you should see** (from the fully trained Chapter 4 version):

**Expected output:**

```
chrmahzlyn
hlri
khmrix
thty
sklassa
jazhnn
fagdryst
kheric
```

[verified]

Compare to Chapter 2's `momasurailezitynn`. These have plausible name structure and sensible lengths. `kheric` and `sklassa` could almost pass. The model still cannot spell, because three characters of memory is not much.

### Exercises

1. **Run the learning-rate sweep yourself.** Wrap the network build and training loop in a function taking `lr`, and reproduce the table in step 5.
2. **Change the context length** from 3 to 5 and to 8. Rebuild the dataset (only `block_size` changes) and note the dev loss each time. Where does it stop helping?
3. **Change the embedding size** from 10 to 2, and to 30. Plot the 2-dimensional version with matplotlib: `plt.scatter(C[:,0].data, C[:,1].data)` and annotate each point with its letter. The vowels should cluster.
4. **Deliberately overfit.** Train the full network on 200 examples for 20,000 steps and watch train loss go near zero while dev loss climbs. That is overfitting, produced on purpose so you recognize it later.

### Troubleshooting

| Symptom | Cause |
|---|---|
| `RuntimeError: shape '[-1, 30]' is invalid` | `block_size` and the `W1` input dimension disagree; the product `n_embd × block_size` must equal `W1.shape[0]` |
| Dev loss much worse than train | Overfitting: shrink the model, add regularization, or get more data |
| Both losses stuck near 2.45 | Your model is doing no better than a bigram; check the context is being fed in (print `Xtr[:5]`) |
| Loss explodes to hundreds | Learning rate too high, as at 10.0 in the sweep |
| Samples are gibberish with no `.` | You forgot to stop generation at index 0 |

### 30-second version

Give each letter ten learned coordinates instead of a slot in a giant table, feed three letters' worth into a small network, and it generalizes to letter combinations it never saw, taking the loss from 2.4544 to 2.2618. The chapter also teaches the craft: mini-batches for speed, finding the learning rate by sweeping it (0.1 works, 10.0 explodes the loss to 87), splitting data three ways by word rather than by example, and proving your model can learn by making it memorize 32 examples first.

---